# X1: Sensitivity, SNR, inner products & likelihoods

**Development-stack exercise notebook (LATW `dev` branch).** There is no
Colab button: these exercises target the *development* versions of the LISA
Analysis Tools packages. Set the environment up by cloning LISAanalysistools
and running its installer (it lays every sibling repo out side by side and
editable-installs the development branches):

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

For the workshop on the **pip-released** packages, use the
[`main` branch](https://github.com/lisa-analysis-tools/LATW/tree/main) instead
(branch policy: `main` &harr; pip releases, `dev` &harr; the `install.sh` stack).

In [ ]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

# The stock LISA noise models evaluate 1/f terms at f = 0 on full FFT grids;
# those bins are masked downstream, so silence the numpy divide warnings.
warnings.filterwarnings("ignore", category=RuntimeWarning)

In this notebook we build up the **analysis basics** that every LISA study
rests on: the noise model (a *sensitivity curve* and the channel *covariance
matrix*), the noise-weighted *inner product*, the *signal-to-noise ratio*
(SNR), and the Gaussian *likelihood*. We do it with the modern `lisatools`
data-analysis objects introduced in the informational notebook
[`02`](../../02_Foundations.ipynb) &mdash; `DomainBase` signals
(`FDSignal`/`WDMSignal`), stock `SensitivityMatrix` builders, and the
`AnalysisContainer` that ties them together. Everything works in the **XYZ**
TDI basis (the stock global-fit convention), stays on the laptop CPU, and runs
in a few minutes.

### How these exercises work

Each exercise is one of two kinds:

- **Task N** &mdash; you *write code* toward a stated goal. In this answer
  notebook the solution cells are filled in; in the generated student notebook
  they are blanked (a whole cell, or just the key solution lines for a
  fill-in-the-blank). Every Task ends with a **Useful documentation:** list
  pointing at the Sphinx API docs and the relevant section of an informational
  notebook.
- **Question** (a `### Question` heading) &mdash; a short *discussion* prompt.
  No code required; the answer sketch here is for the group conversation and is
  removed in the student notebook.

The tasks build on each other in order, so run them top to bottom.

## Task 1: Plot the LISA sensitivity curve

LISA's sensitivity to a gravitational wave depends strongly on frequency.
Evaluate the sky-/polarization-averaged strain sensitivity
[`LISASens`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.LISASens) on a
log-spaced frequency grid and plot it as a **characteristic strain**
(`return_type="char_strain"`). Compare **two noise models** &mdash; the
`"sangria"` model used for the LDC Sangria dataset and the Science Requirements
Document model `"scirdv1"` &mdash; on the same axes, and overlay a single TDI
channel curve
[`X1TDISens`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.X1TDISens) (the X
channel; the workshop default is XYZ) so you can see the difference between the
abstract strain sensitivity and a concrete TDI-channel PSD.

Useful documentation:
* [`get_sensitivity`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.get_sensitivity)
* [`LISASens`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.LISASens) /
  [`X1TDISens`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.X1TDISens)
* [`get_available_default_lisa_models`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/detector.html#lisatools.detector.get_available_default_lisa_models)
* Informational notebook: see [`02` &sect; Sensitivity](../../02_Foundations.ipynb)

In [ ]:
# imports
from lisatools.sensitivity import get_sensitivity, LISASens, X1TDISens

### Question

Read the shape of the curve. Where is LISA most sensitive (the bottom of the
"bucket"), and what climbs the sensitivity back up at **low** and at **high**
frequency? Map that shape onto the source classes: which sources sit in the
sensitive bucket and are therefore *loud*, and which are pushed toward the noisy
edges?

*Discussion.* The bucket bottoms out around a few mHz. At **low** frequency the
curve rises steeply because test-mass acceleration noise dominates and the
finite arm cannot resolve very long wavelengths; at **high** frequency it rises
because the GW wavelength becomes shorter than the arm and the response rolls
off (plus optical-metrology noise). Massive black-hole binaries and the bulk of
the galactic binaries live near the sensitive few-mHz region and are loud;
very-low-frequency sources and anything above ~0.1 Hz are strongly suppressed.
Swapping `sangria` for `scirdv1` mostly shifts the noise levels, which rescales
every source's SNR without moving where the sensitive band sits.

## Task 2: Build the XYZ channel-covariance matrix

A single curve is enough to eyeball a source, but the likelihood needs the
full **channel covariance** at every frequency. The three TDI channels X, Y, Z
share noise sources, so their covariance is a dense 3&times;3 Hermitian matrix
per frequency &mdash; PSDs on the diagonal, cross-spectra (CSDs) off it. Build a
[`XYZ2SensitivityMatrix`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.XYZ2SensitivityMatrix)
(TDI-2) on an
[`FDSettings`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSettings) grid, print the
shapes of the matrix and of the two quantities the Gaussian likelihood consumes
&mdash; `invC` (the inverse covariance) and `detC` (its determinant) &mdash;
and plot a diagonal PSD (`S_XX`) against an off-diagonal CSD (`S_XY`) to see
their relative size.

Useful documentation:
* [`XYZ2SensitivityMatrix`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.XYZ2SensitivityMatrix)
* [`SensitivityMatrix`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.SensitivityMatrix)
* [`FDSettings`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSettings)
* Informational notebook: see [`02` &sect; the channel covariance `SensitivityMatrix`](../../02_Foundations.ipynb)

In [ ]:
# imports
from lisatools.sensitivity import XYZ2SensitivityMatrix
from lisatools.domains import FDSettings

## Task 3: Inject a galactic binary and compute its inner product & SNR

Now inject a real source. Use
[`gbgpu`](https://github.com/mikekatz04/GBGPU)'s `GBGPU` to generate one
galactic binary as a TDI-2 **XYZ** response on the full rFFT grid, wrap it in an
[`FDSignal`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSignal) band-limited around
the source, pair it with an `XYZ2SensitivityMatrix`, and combine the two in an
[`AnalysisContainer`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#analysis-container) &mdash; the
"atom" of every LISA analysis. Read off the noise-weighted inner product
`<d|d>` (`.inner_product()`) and the optimal SNR `sqrt(<d|d>)` (`.snr()`).

The GB generator below is **provided** (it is the `gbgpu` recipe from
notebook [`02`](../../02_Foundations.ipynb)); run it as-is. Your task is the
following cell: band-limit the data into an `FDSignal`, build the
`XYZ2SensitivityMatrix`, drop both into an `AnalysisContainer`, and read off
`<d|d>` and the SNR.

Then compute the **same** inner product by hand with NumPy and confirm the two
agree. Because the XYZ channels are correlated, the by-hand formula contracts
the data with the **inverse** covariance `invC`:

$$\langle d | d \rangle = 4\,\Delta f\;\mathrm{Re}\sum_f \sum_{i,j}\; \tilde d_i(f)^{*}\,\big(C^{-1}\big)_{ij}(f)\,\tilde d_j(f)\ .$$

Useful documentation:
* [`AnalysisContainer`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#analysis-container) /
  [`.inner_product`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.inner_product) /
  [`.snr`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.snr)
* [`FDSignal`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSignal)
* [`inner_product`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/diagnostic.html#lisatools.diagnostic.inner_product)
* Informational notebook: see [`02` &sect; the `AnalysisContainer`](../../02_Foundations.ipynb)

In [ ]:
# imports
from gbgpu.gbgpu import GBGPU
from lisatools.detector import EqualArmlengthOrbits
from lisatools.domains import FDSignal
from lisatools.analysiscontainer import AnalysisContainer

In [ ]:
# provided helper (from notebook 02): run this cell as-is
# --- grid (chosen so the exact same samples also form a WDM grid in Task 5) ---
dt = 15.0
Nf_w, Nt_w = 128, 512          # WDM layers x time pixels (used in Task 5)
Ng = Nf_w * Nt_w               # total time samples
Tg = Ng * dt                   # observation span
data_length = Ng // 2 + 1      # rFFT length
df = 1.0 / Tg

gb = GBGPU(force_backend="cpu", orbits=EqualArmlengthOrbits(force_backend="cpu"))
gb.gpus = None                 # generate_global_template reads this; None = CPU

# amp, f0, fdot, phi0, iota, psi, ecliptic lon, ecliptic lat
truth = np.array([2e-22, 3.0e-3, 1.0e-17, 0.3, 0.7, 1.2, 2.0, 0.5])

def gb_fd_xyz(A, f0, fdot, phi0, iota, psi, lam, beta, oversample=4):
    """One GB's TDI-2 XYZ response on the full rFFT grid -> (3, data_length)."""
    params = np.atleast_2d([A, f0, fdot, 0.0, phi0, iota, psi, lam, beta]).astype(np.float64)
    buf = np.zeros(3 * data_length, dtype=np.complex128)
    gb.generate_global_template(params, np.zeros(params.shape[0], dtype=np.int32), buf,
        start_freq_ind=0, T=Tg, dt=dt, tdi_channel_setup="XYZ", tdi2=True,
        oversample=oversample, data_length=data_length)
    return buf.reshape(3, data_length)

injection = gb_fd_xyz(*truth)
print("injection shape:", injection.shape, " nonzero X bins:", int(np.count_nonzero(injection[0])))

Now the by-hand cross-check. Fill in the single line that evaluates the
inner-product formula above (contract the data with `invC` and take
`4 df Re[...]`); the surrounding scaffolding is already written for you.

In [ ]:
d = np.asarray(data.arr)          # (3, N_band)
invC = np.asarray(sens.invC)      # (3, 3, N_band): inverse covariance per freq

# <d|d> = 4 df  Re  sum_f sum_ij  d_i^*  invC_ij  d_j

print(f"by-hand   <d|d> = {dd_manual:.4f}")
print(f"container <d|d> = {ac.inner_product():.4f}")
print(f"optimal SNR     = {np.sqrt(dd_manual):.4f}")

## Task 4: The Gaussian likelihood at the truth and at an offset

The LISA likelihood is Gaussian in the residual $r = d - h$:
$\ln\mathcal{{L}} = -\tfrac12\langle d-h\,|\,d-h\rangle$ (up to the noise
normalization). Evaluate it with
[`AnalysisContainer.template_likelihood`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.template_likelihood)
for two templates: the **true** parameters (the residual vanishes, so
$\ln\mathcal{{L}}\approx 0$) and a slightly **offset** set (nudge $f_0$ by a
fraction of a frequency bin and shift the phase). Print the two log-likelihoods
and their difference, and use
[`.template_snr`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.template_snr)
to watch the *detected* SNR of the offset template fall below the optimal SNR as
the template decorrelates from the data.

Useful documentation:
* [`.template_likelihood`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.template_likelihood)
* [`.template_snr`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.template_snr)
* Informational notebook: see [`02` &sect; the `AnalysisContainer`](../../02_Foundations.ipynb)

In [ ]:
# imports
from lisatools.domains import FDSignal

### Question

A tiny shift in $f_0$ &mdash; a *fraction* of one frequency bin &mdash;
already drops the log-likelihood by a lot. What does the **width** of the
likelihood (how fast $\ln\mathcal{L}$ falls as you leave the truth) tell you
about how precisely a parameter can be *measured*?

*Discussion.* Near the peak the log-likelihood is approximately a downward
parabola, $\ln\mathcal{L} \approx -\tfrac12 (\Delta\theta/\sigma_\theta)^2$,
so a *narrow* likelihood (steep fall) means a *small* measurement uncertainty
$\sigma_\theta$. The curvature that sets that width scales with the squared
SNR, so louder sources are pinned down more tightly. Frequency is measured
extraordinarily well for a long-lived monochromatic source because the phase
accumulates over the whole observation: even a sub-bin shift in $f_0$ dephases
the template across $T_\text{obs}$ and destroys the overlap. This is exactly
the curvature the Fisher/information matrix captures (informational notebook
[`02` &sect; Fisher information](../../02_Foundations.ipynb)).

## Task 5: The same SNR in a different basis (WDM wavelets)

The inner product and SNR are properties of the *signal and the noise*, not
of the basis you happen to compute them in. Demonstrate this by recomputing the
**same** GB's `<d|d>` and SNR in the **WDM wavelet** time-frequency basis and
checking it matches the frequency-domain answer from Task 3.

Transform the full-band FD injection into the wavelet basis with
[`FDSignal.wdmtransform`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSignal.wdmtransform)
onto a
[`WDMSettings`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.WDMSettings) grid (the grid
was chosen in Task 3 so that `Nf_w * Nt_w` equals the number of time samples),
band-limited to the same frequency window. Build an `XYZ2SensitivityMatrix` on
that **same** WDM grid (the noise folds consistently into the wavelet basis),
drop both into a new `AnalysisContainer`, and compare.

Useful documentation:
* [`WDMSettings`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.WDMSettings) /
  [`FDSignal.wdmtransform`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/domains.html#lisatools.domains.FDSignal.wdmtransform)
* [`XYZ2SensitivityMatrix`](https://mikekatz04.github.io/LISAanalysistools/build/html/user/sensitivity.html#lisatools.sensitivity.XYZ2SensitivityMatrix)
* Informational notebook: see [`02` &sect; Domains](../../02_Foundations.ipynb)
  and [`02` &sect; the WDM wavelet grid](../../02_Foundations.ipynb)

In [ ]:
# imports
from lisatools.domains import WDMSettings, WDMSignal

The two SNRs agree to about a percent. The small residual is the wavelet
*discretization* (a finite `Nf`&times;`Nt` grid) plus the fold that maps the
frequency-domain noise PSD into the wavelet layers &mdash; not a physical
difference. Refining the grid tightens the agreement.

### Question

Why *should* the SNR come out (essentially) the same in the frequency domain
and in the wavelet domain? What has to be true of the transform and of how the
noise is treated for this to hold?

*Discussion.* The SNR is $\sqrt{\langle h|h\rangle}$ with the inner product
weighted by the noise covariance. Moving between the frequency basis and the WDM
wavelet basis is (up to discretization) a **unitary change of basis**, and the
noise PSD is transformed *consistently* into the new basis (the "fold"). Under a
unitary map a noise-weighted inner product is invariant &mdash; this is the
Parseval/Plancherel statement generalised to a colored-noise metric &mdash; so
the SNR is basis-independent. That freedom is what lets the global fit pick
whichever basis is *cheapest* for a given source: narrow-band galactic binaries
and stellar-origin binaries are scored in the sparse WDM/wavelet basis, while
other sources stay in the frequency domain, all feeding one consistent
likelihood.

### Where this goes next

You now have the analysis atom: a noise model, an inner product, an SNR, and a
likelihood, all computed through the `AnalysisContainer`. The next exercise
notebook, [`X3` Fixed-dimensional MCMC with Eryn](X3_FixedDimMCMC.ipynb), turns
that single-point likelihood into a *posterior* by sampling &mdash; and the very
container you built here becomes the sampler's log-likelihood (see the
informational notebook [`05`](../../05_ErynSmallToLarge.ipynb)).